# Exercise 1

--> courseutils.py in same folder (great idea for own projects as well!!!)

## Need to make a new environment since gensim is not compatible with they numpy version we need elsewhere in the course 

before you proceed: copy your existing environment, activate the new environment and then pip install gensim (in that new environment (!), f.e. called 'gesis_iml_gensim') 

In [ ]:
from nltk.tokenize import sent_tokenize
import string
import re

# tqdm allows you to display progress bars in loops
from tqdm import tqdm
from datetime import datetime

# you need to have courseutils.py in the same folder
from courseutils import get_review_data

import gensim

# lets get more output
import logging
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

note that there are some slight syntax changes between gensim 3 and 4; notebook is now optimized for gensim 4.

In [ ]:
gensim.__version__

## Step 1: Get a lot of texts

I'll just take the movie reviews here, but you are *very much encouraged* to take your own data. Use any method to get them into a long list (or similar).

In [ ]:
train, test, _, _ = get_review_data()

In [ ]:
# we just need one list
print(f"The original dataset has two sets of reviews of length {len(train)} and {len(test)}")
train.extend(test)
del test
print(f"We merged them into one list of {len(train)} reviews")

In [ ]:
train[:3]

## Step 2: Reformat
We want to train on sentences, not on whole reviews. We don't need a list of reviews, but a list of sentences.

Also, **we only want unique sentences**. It has been shown that this improves the resulting models (and it speeds up training, of course).

There are different ways of achieving this, here is one. Some remarks:

- tqdm displays a progress bar - it's not strictly necessary
- a set is like a list without order, and all items are guaranteed to be unique. You could also use a list, but this is faster. Then, you need to use `uniquesentences = []` and `.append()` instead of `.add()`
- we also remove punctuation 
- depending on whether the texts we want to use our model on later on are lowercased or not, we have to (or not) lowercase here as well. That's a decision to make.

In [ ]:
trans = str.maketrans('', '', string.punctuation) # translation scheme for removing punctuation
uniquesentences = set()
for review in tqdm(train):
    for sentence in sent_tokenize(review):
        # remove HTML tags in there
        sentence = re.sub(r"<.*?>"," ",sentence)
        sentence = sentence.translate(trans) 
        if sentence not in uniquesentences:
            uniquesentences.add(sentence.lower())

In [ ]:
print(f"We now have {len(uniquesentences)} unique sentences.")

In [ ]:
# if we want to, we can turn the set into a list and expect it, e.g. like this:
# list(uniquesentences)[:10]

**Note that uniquesentences can be also a generator that reads from disk (or from elsewhere) for the next step. Hence, it is possible to train models on more sentences than fit in your memory!**

## Step 3: Train the model

That's really straightforward in gensim

In [ ]:
# we do not need a list of lists of tokens later on, so let's use a generator instead of a list to save memory
# note that we use round parentheses instead of square brackets to achieve this
# we do need two generators, though, as we first need to build the vocabulary and later need to train.
# If we use a list, we obviously only need once.
tokenizedsentences = (sentence.split() for sentence in uniquesentences)
tokenizedsentences2 = (sentence.split() for sentence in uniquesentences)

In [ ]:
print(f"Started setting up the model at {datetime.now()}")
model = gensim.models.Word2Vec(vector_size=300) # we want 300 dimensions
model.build_vocab(tokenizedsentences)
print(f"Started training at {datetime.now()}")
model.train(tokenizedsentences2, total_examples=model.corpus_count,  epochs=1)
# our model gets better if we use more epochs, but we can only do so if we use a list instead of a generator as input
# after all, you can only pass over a generator once.
# model.train(tokenizedsentences2, total_examples=model.corpus_count,  epochs=model.epochs)
print(f"Finished training at {datetime.now()}")

In [ ]:
gensim.models.Word2Vec?

In [ ]:
model.save("mymodel")

In [ ]:
# and load it again, just to check
mymodel = gensim.models.Word2Vec.load("mymodel")

# Step 4: Play with the model

In [ ]:
animals = ['cat', 'dog', 'horse', 'goldfish', 'lion']
for animal in animals:
    try:
        print(f"A {animal} is almost the same as a {model.wv.most_similar(animal)[0][0]}.")
    except Exception as e:
        print(e)

In [ ]:
animals = ['director', 'actor', 'bad', 'good']
for animal in animals:
    try:
        print(f"A {animal} is almost the same as a {model.wv.most_similar(animal)[0][0]}.")
    except Exception as e:
        print(e)

In [ ]:
model.wv.most_similar("action")

In [ ]:
model.wv.most_similar("movie")

In [ ]:
# the classic king/queen example (of course this one doesn't necessarily work out on movie-trained embeddings)
model.wv.most_similar(positive=["king", "woman"], negative=["man"])

# 5 Adapt

Now it's time to dive into the gensim documentation (online or via `?` / tab completion) to figure out the options you have - e.g., skipgram vs CBOW, dimensions, etc.

# Other things we can do

Some things that came up.


## Dictionary expansion

One cool thing that we can do is to expand a list of words with near-synonyms. For instance, imagine we use a simple keyword-based approach to count how often some word we are interested in occurs in a corpus. But maybe we forgot one? In that case, why not expand our list with (near-)synonyms from an embedding model?

In [ ]:
# let's get the most similar words to our seed words
seedlist = ['action', 'horror', 'comedy','nature','family','cartoon']
words = seedlist.copy()
for w in seedlist:
    words.extend([e[0] for e in model.wv.most_similar(w)])

print(words)

In [ ]:
[e[0] for e in model.wv.most_similar("action")]

## Plotting

We can also use a method called tsne (from our good friend scikit-learn) to make a two-dimensional projection of our 300-dimensional embeddings to plot the distances of the words (not the most interesting data here, but you get the idea):

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

In [ ]:
def plot_words(words, model):
    '''takes a list of words and a word embedding model as input and plots the words
    in a 2-dimensional projection of the space'''
    # make sure words are unique:
    words = set(words)
    X = model.wv[words]
    tsne = TSNE(n_components=2)
    X_tsne = tsne.fit_transform(X)

    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)
    ax.scatter(X_tsne[:, 0], X_tsne[:, 1])
    for w, pos in zip(words,X_tsne):
        ax.annotate(w, pos)

In [ ]:
plot_words(words, model)
